# Training model probabilistik semifinal

Notebook ini mengendalikan audit data, chronological cross-validation, training final, dan pembuatan artifact. Target skor 90 menit hanya memakai pertandingan berlabel `Regular`; laga AET dan penalti tidak diperlakukan sebagai label skor 90 menit.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import RANDOM_SEED
from src.data import build_match_features, load_raw_data
from src.training import run_training

print(f'Python {sys.version.split()[0]} | seed {RANDOM_SEED} | root {ROOT}')

Python 3.13.4 | seed 20260713 | root /home/user/projects/world-cup-predict


## Audit sumber data

Output hanya memuat jumlah baris dan kolom. Raw CSV tidak disalin ke notebook atau repository.

In [2]:
raw = load_raw_data()
pd.DataFrame([
    {'tabel': name, 'baris': len(frame), 'kolom': len(frame.columns)}
    for name, frame in raw.items()
]).sort_values('tabel').reset_index(drop=True)

,tabel,baris,kolom
0,events,790,6
1,matches,104,17
2,players,1248,21
3,referees,28,4
4,squads,1248,10
5,stages,7,3
6,stats,200,12
7,teams,48,8
8,venues,16,8


In [3]:
features = build_match_features(raw)
audit = {
    'pertandingan_dengan_tim': len(features),
    'selesai': int((features.status == 'Completed').sum()),
    'regular_target_90': int(((features.status == 'Completed') & (features.result_type == 'Regular')).sum()),
    'AET_dikeluarkan': int((features.result_type == 'AET').sum()),
    'penalti_dikeluarkan': int((features.result_type == 'Penalties').sum()),
}
pd.Series(audit, name='jumlah')

pertandingan_dengan_tim    102
selesai                    100
regular_target_90           92
AET_dikeluarkan              4
penalti_dikeluarkan          4
Name: jumlah, dtype: int64

## Training, evaluasi, dan artifact

Pemilihan model memakai tiga expanding-window folds. Model kandidat hanya digunakan bila exact-score log loss atau predictive count likelihood mengungguli baseline terkait.

In [4]:
predictions, evaluation = run_training()
score_summary = pd.DataFrame(evaluation['score']['aggregate']).T
score_summary.index.name = 'model'
score_summary

,exact_score_log_loss,outcome_log_loss
model,,
smoothed_baseline,3.213447,1.068018
regularized_poisson,2.923547,0.769670
poisson_dc_blend,2.921421,0.775101
dixon_coles,2.923727,0.781909


In [5]:
pd.DataFrame([
    {
        'pertandingan': f"{match['home_team']} vs {match['away_team']}",
        'xG_home': match['score_90']['expected_goals']['home'],
        'xG_away': match['score_90']['expected_goals']['away'],
        'home_win': match['outcome_90']['home_win'],
        'draw': match['outcome_90']['draw'],
        'away_win': match['outcome_90']['away_win'],
        'shootout': match['shootout_probability'],
    }
    for match in predictions['matches']
]).round(4)

,pertandingan,xG_home,xG_away,home_win,draw,away_win,shootout
0,France vs Spain,1.1695,1.6739,0.2585,0.2581,0.4834,0.1230
1,England vs Argentina,1.2908,1.5305,0.3127,0.2665,0.4208,0.1282


In [6]:
pd.DataFrame([
    {
        'target': target,
        'model': metrics['selected_model'],
        'fallback': metrics['fallback_used'],
    }
    for target, metrics in evaluation['events'].items()
])

,target,model,fallback
0,shots,regularized_negative_binomial,False
1,shots_on_target,regularized_poisson,False
2,corners,regularized_poisson,False
3,fouls,smoothed_baseline,True
4,offsides,regularized_poisson,False
5,yellow_cards,regularized_poisson,False
6,saves,regularized_poisson,False


## Batas interpretasi

Probabilitas adalah estimasi dari sampel turnamen yang kecil, bukan kepastian. Red card dan VAR hanya memakai probabilitas tersmoothing dan selalu diberi confidence rendah. Peluang menang shootout dibagi 50/50 karena data tidak cukup untuk mengestimasi kemampuan penalti per tim. Artifact sudah melewati pemeriksaan normalisasi, nilai finite, non-negativity, dan orientasi home/away.